# CV面试专题 02: 滤波与卷积

本节涵盖计算机视觉面试中常见的滤波与卷积知识点:
- 卷积操作原理
- 均值/高斯/中值/双边滤波
- 边缘检测算子 (Sobel, Laplacian)
- 可分离滤波优化
- 频域处理基础

**使用方法**: 在每个题目 cell 中修改 `answer` 变量,然后运行 cell 检查答案。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import timeit

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
print('OpenCV version:', cv2.__version__)

## 一、选择题 (12题)

In [ ]:
# 题目 1: cv2.filter2D 参数含义
print("""
Q: cv2.filter2D(src, ddepth, kernel) 中, ddepth=-1 表示什么?
A) 输出图像深度与输入相同
B) 输出图像深度为浮点型
C) 不进行卷积操作
D) 使用默认的 3x3 卷积核
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "A"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: ddepth=-1 表示输出图像深度与输入相同。若输入是 uint8, 输出也是 uint8。注意: Sobel 等梯度计算可能产生负值, 需要设为 cv2.CV_64F 避免截断。")

In [ ]:
# 题目 2: 均值滤波 vs 高斯滤波
print("""
Q: 均值滤波和高斯滤波的主要区别是什么?
A) 均值滤波更快, 高斯滤波更慢
B) 均值滤波权重均匀, 高斯滤波中心权重大、边缘小
C) 高斯滤波可以去除椒盐噪声, 均值滤波不能
D) 两者效果完全相同
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: 均值滤波核内所有像素权重相等 (1/N), 高斯滤波按高斯分布赋权重, 中心像素权重最大, 距离越远权重越小。高斯滤波在平滑的同时更好地保留图像特征。")

In [ ]:
# 题目 3: 高斯滤波 sigma 参数
print("""
Q: 高斯滤波器中, sigma 值增大时, 模糊效果会怎样?
A) 模糊程度减弱
B) 模糊程度增强
C) 没有变化
D) 只影响边缘区域
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: sigma 越大, 高斯分布越平坦, 远处像素的权重增大, 参与平均的像素更多, 模糊程度越强。cv2.GaussianBlur 中若 sigmaX=0, 则 sigma 从核大小自动计算: sigma = 0.3*((ksize-1)*0.5 - 1) + 0.8")

In [ ]:
# 题目 4: 中值滤波适用场景
print("""
Q: 中值滤波特别适合去除哪种噪声?
A) 高斯噪声
B) 椒盐噪声
C) 泊松噪声
D) 均匀噪声
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: 椒盐噪声表现为随机出现的极端像素值 (纯黑/纯白), 中值滤波取邻域中值, 能有效排除这些极端值。而均值滤波会被极端值拉偏, 高斯滤波也无法完全消除。这是面试超高频题!")

In [ ]:
# 题目 5: 双边滤波特点
print("""
Q: 双边滤波相比高斯滤波的最大优势是什么?
A) 计算速度更快
B) 同时考虑空间距离和像素值差异, 能保边去噪
C) 可以处理彩色图像
D) 核大小可以任意调整
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: 双边滤波有两个权重: 空间高斯权重 (距离越近权重越大) + 值域高斯权重 (像素值越接近权重越大)。在边缘处像素值差异大, 值域权重小, 因此边缘得以保留。缺点: 速度慢, 参数多 (d, sigmaColor, sigmaSpace)。")

In [ ]:
# 题目 6: Sobel 算子方向
print("""
Q: 使用 Sobel 算子检测垂直边缘 (即左右方向的灰度变化), 应该使用哪个方向的核?
A) 水平方向 (dx=1, dy=0)
B) 垂直方向 (dx=0, dy=1)
C) 两个方向都需要
D) 用 Laplacian 即可
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "A"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: 检测垂直边缘 = 检测水平方向(x方向)的灰度变化, 用 dx=1, dy=0。Sobel X核在x方向做差分。检测水平边缘用 dx=0, dy=1。注意: 方向和检测的边缘方向是垂直关系, 容易混淆!")

In [ ]:
# 题目 7: 拉普拉斯算子
print("""
Q: 拉普拉斯算子是几阶导数算子?
A) 一阶导数
B) 二阶导数
C) 三阶导数
D) 零阶 (无导数)
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: 拉普拉斯算子是二阶导数算子: d²f/dx² + d²f/dy²。对噪声非常敏感, 通常先高斯平滑再用拉普拉斯 (即 LoG 算子)。常用 3x3 核: [[0,1,0],[1,-4,1],[0,1,0]]")

In [ ]:
# 题目 8: cv2.blur vs cv2.GaussianBlur
print("""
Q: cv2.blur(img, (5,5)) 和 cv2.GaussianBlur(img, (5,5), 0) 的参数区别是什么?
A) blur 只需要核大小, GaussianBlur 还需要 sigma 参数
B) blur 使用固定权重, GaussianBlur 使用高斯权重
C) 两者的核大小参数含义不同
D) A 和 B 都正确
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "D"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: cv2.blur 的核中所有元素相等 (均匀权重 1/N), 只需指定核大小。cv2.GaussianBlur 除了核大小还需要 sigma 参数控制高斯分布形状。两者核大小参数含义相同 (宽,高), 但权重分配方式不同。所以 A 和 B 都正确。")

In [ ]:
# 题目 9: 卷积核大小对性能的影响
print("""
Q: 使用 7x7 卷积核处理 1000x1000 图像, 与 3x3 核相比, 计算量大约增加多少倍?
A) 约 2.3 倍
B) 约 5.4 倍
C) 约 10 倍
D) 没有明显变化
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: 每个像素的计算量与核大小成正比。7x7=49 次乘加, 3x3=9 次乘加。49/9 ≈ 5.4 倍。卷积计算量 O(H*W*K*K), K 为核大小。面试中常考计算量估算。")

In [ ]:
# 题目 10: 可分离滤波
print("""
Q: 一个 2D 高斯核是可分离的, 分解为两个 1D 核后, 计算复杂度从 O(N²) 变为?
A) O(N)
B) O(2N)
C) O(N²/2)
D) O(log N)
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: NxN 的 2D 卷积核可分解为先做 N 次 1D 水平卷积, 再做 N 次 1D 垂直卷积。每个像素从 N² 次运算变为 2N 次运算。对于 7x7 核: 49 → 14, 加速约 3.5 倍!")

In [ ]:
# 题目 11: 卷积的边界处理
print("""
Q: cv2.filter2D 默认的边界填充方式 (borderType) 是什么?
A) 零填充 (BORDER_CONSTANT, 填0)
B) 边界复制 (BORDER_REPLICATE)
C) 反射填充 (BORDER_REFLECT)
D) 环绕填充 (BORDER_WRAP)
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: cv2.filter2D 默认使用 BORDER_REPLICATE (边界复制), 即用最边缘的像素值填充。其他选项: BORDER_CONSTANT 用固定值填充, BORDER_REFLECT 做镜像反射。不同填充方式对边缘区域结果有影响。")

In [ ]:
# 题目 12: 卷积与相关的区别
print("""
Q: 严格数学意义上, cv2.filter2D 执行的是卷积(convolution) 还是互相关(correlation)?
A) 卷积 (需要翻转核)
B) 互相关 (不翻转核)
C) 取决于 ddepth 参数
D) 两者交替执行
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("解析: cv2.filter2D 实际执行的是互相关操作 (不翻转核)。严格卷积需要将核旋转180度后再做互相关。对于对称核 (如高斯、均值), 卷积和互相关结果相同。但对于 Sobel 等非对称核, 结果会不同。")

## 二、编程练习

### 练习 1: 手动实现 2D 卷积

In [ ]:
# TODO: 不使用 cv2.filter2D, 手动实现 2D 卷积操作
# 要求: 支持任意大小的核, 使用零填充

def conv2d_manual(image, kernel):
    """
    手动实现 2D 卷积
    
    Args:
        image: 灰度图像, shape (H, W)
        kernel: 卷积核, shape (kH, kW)
    Returns:
        output: 卷积结果, shape (H, W)
    """
    # TODO: 在此实现
    pass

# 简单测试
test_img = np.array([[1, 2, 3, 4],
                     [5, 6, 7, 8],
                     [9, 10, 11, 12]], dtype=np.float64)
kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]], dtype=np.float64)  # Sobel X

result = conv2d_manual(test_img, kernel)
print(result)

In [ ]:
# ====== 参考答案 ======

def conv2d_manual(image, kernel):
    """手动实现 2D 卷积 (实际上是互相关, 与 cv2.filter2D 一致)"""
    h, w = image.shape
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    
    # 零填充
    padded = np.zeros((h + 2 * pad_h, w + 2 * pad_w), dtype=np.float64)
    padded[pad_h:pad_h+h, pad_w:pad_w+w] = image
    
    output = np.zeros_like(image, dtype=np.float64)
    
    # 滑动窗口卷积
    for i in range(h):
        for j in range(w):
            region = padded[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * kernel)
    
    return output

# 测试
test_img = np.array([[1, 2, 3, 4],
                     [5, 6, 7, 8],
                     [9, 10, 11, 12]], dtype=np.float64)
kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]], dtype=np.float64)

result_manual = conv2d_manual(test_img, kernel)
result_cv2 = cv2.filter2D(test_img, cv2.CV_64F, kernel)

print('手动结果:\n', result_manual)
print('cv2结果:\n', result_cv2)
print('差异:', np.abs(result_manual - result_cv2).max())

### 练习 2: 手动构建高斯核并应用

In [ ]:
# TODO: 不使用 cv2.getGaussianKernel, 手动构建 2D 高斯核

def make_gaussian_kernel(size, sigma):
    """
    手动构建 2D 高斯核
    
    Args:
        size: 核大小 (奇数)
        sigma: 标准差
    Returns:
        kernel: 归一化的高斯核, shape (size, size)
    """
    # TODO: 在此实现
    pass

# 测试
k = make_gaussian_kernel(5, 1.0)
print('高斯核 (5x5, sigma=1.0):')
print(np.round(k, 4))
print('核总和:', k.sum())

In [ ]:
# ====== 参考答案 ======

def make_gaussian_kernel(size, sigma):
    """手动构建 2D 高斯核"""
    # 方法: 先构建 1D 高斯, 再通过外积得到 2D 核
    ax = np.arange(size) - size // 2  # [-2, -1, 0, 1, 2] for size=5
    
    # 1D 高斯
    gauss_1d = np.exp(-ax**2 / (2 * sigma**2))
    
    # 2D 高斯 = 1D 外积 1D (利用可分离性)
    kernel = np.outer(gauss_1d, gauss_1d)
    
    # 归一化
    kernel /= kernel.sum()
    return kernel

# 验证
k_manual = make_gaussian_kernel(5, 1.0)
print('手动高斯核:\n', np.round(k_manual, 4))
print('核总和:', k_manual.sum())

# 与 OpenCV 对比
k_cv2_1d = cv2.getGaussianKernel(5, 1.0)
k_cv2 = k_cv2_1d @ k_cv2_1d.T
print('\nOpenCV 高斯核:\n', np.round(k_cv2, 4))
print('差异:', np.abs(k_manual - k_cv2).max())

# 应用到测试图像
test_img = np.random.randint(0, 256, (100, 100), dtype=np.uint8)
result_manual = cv2.filter2D(test_img, -1, (k_manual * 255).astype(np.float32) / 255)
result_cv2 = cv2.GaussianBlur(test_img, (5, 5), 1.0)
print(f'\n应用结果差异: {np.abs(result_manual.astype(int) - result_cv2.astype(int)).mean():.2f} (平均)')

### 练习 3: 手动实现 Sobel 边缘检测

In [ ]:
# TODO: 手动构建 Sobel 核, 实现 Sobel 边缘检测
# 步骤: 分别计算 X 和 Y 方向梯度, 然后合成梯度幅值

def sobel_edge_detection(gray_img):
    """
    使用手动构建的 Sobel 核进行边缘检测
    
    Args:
        gray_img: 灰度图像, uint8
    Returns:
        gradient_mag: 梯度幅值图, uint8
        grad_x: X方向梯度
        grad_y: Y方向梯度
    """
    # TODO: 在此实现
    pass

# 测试
np.random.seed(42)
test_img = np.zeros((100, 100), dtype=np.uint8)
test_img[20:80, 20:80] = 255  # 白色方块

mag, gx, gy = sobel_edge_detection(test_img)
print(f'梯度幅值范围: [{mag.min()}, {mag.max()}]')

In [ ]:
# ====== 参考答案 ======

def sobel_edge_detection(gray_img):
    """手动构建 Sobel 核进行边缘检测"""
    # Sobel X 核 (检测垂直边缘)
    sobel_x = np.array([[-1, 0, 1],
                        [-2, 0, 2],
                        [-1, 0, 1]], dtype=np.float64)
    
    # Sobel Y 核 (检测水平边缘)
    sobel_y = np.array([[-1, -2, -1],
                        [ 0,  0,  0],
                        [ 1,  2,  1]], dtype=np.float64)
    
    # 转为 float 避免截断
    img_float = gray_img.astype(np.float64)
    
    # 计算两个方向的梯度
    grad_x = cv2.filter2D(img_float, cv2.CV_64F, sobel_x)
    grad_y = cv2.filter2D(img_float, cv2.CV_64F, sobel_y)
    
    # 梯度幅值 = sqrt(gx² + gy²)
    gradient_mag = np.sqrt(grad_x**2 + grad_y**2)
    
    # 归一化到 [0, 255]
    gradient_mag = np.clip(gradient_mag, 0, 255).astype(np.uint8)
    
    return gradient_mag, grad_x, grad_y

# 测试
test_img = np.zeros((100, 100), dtype=np.uint8)
test_img[20:80, 20:80] = 255

mag, gx, gy = sobel_edge_detection(test_img)

# 与 OpenCV Sobel 对比
gx_cv2 = cv2.Sobel(test_img, cv2.CV_64F, 1, 0, ksize=3)
gy_cv2 = cv2.Sobel(test_img, cv2.CV_64F, 0, 1, ksize=3)
mag_cv2 = np.sqrt(gx_cv2**2 + gy_cv2**2)
mag_cv2 = np.clip(mag_cv2, 0, 255).astype(np.uint8)

print(f'梯度幅值范围: [{mag.min()}, {mag.max()}]')
print(f'与 OpenCV 差异: {np.abs(mag.astype(int) - mag_cv2.astype(int)).max()}')

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(test_img, cmap='gray'); axes[0].set_title('原图')
axes[1].imshow(np.abs(gx).astype(np.uint8), cmap='gray'); axes[1].set_title('X 梯度 (垂直边缘)')
axes[2].imshow(np.abs(gy).astype(np.uint8), cmap='gray'); axes[2].set_title('Y 梯度 (水平边缘)')
axes[3].imshow(mag, cmap='gray'); axes[3].set_title('梯度幅值')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 练习 4: 可分离滤波效率对比

In [ ]:
# TODO: 对比 2D 卷积 vs 可分离的两次 1D 卷积的性能差异
# 使用 timeit 计时, 对较大的图像测试

def separable_filter_2d(image, kernel_1d):
    """
    使用可分离滤波: 先水平后垂直
    
    Args:
        image: 灰度图像
        kernel_1d: 1D 核
    Returns:
        result: 滤波结果
    """
    # TODO: 在此实现
    pass

def direct_filter_2d(image, kernel_2d):
    """直接 2D 卷积"""
    return cv2.filter2D(image, cv2.CV_64F, kernel_2d)

print('请实现 separable_filter_2d 函数')

In [ ]:
# ====== 参考答案 ======

def separable_filter_2d(image, kernel_1d):
    """可分离滤波: 先水平后垂直"""
    k = kernel_1d.reshape(1, -1)  # 水平 1D 核
    temp = cv2.filter2D(image, cv2.CV_64F, k)  # 水平卷积
    k = kernel_1d.reshape(-1, 1)  # 垂直 1D 核
    result = cv2.filter2D(temp, cv2.CV_64F, k)  # 垂直卷积
    return result

# 准备测试数据
test_img = np.random.randint(0, 256, (500, 500), dtype=np.uint8).astype(np.float64)

# 构建高斯核
size = 15
sigma = 3.0
k_1d = cv2.getGaussianKernel(size, sigma).flatten()
k_2d = np.outer(k_1d, k_1d)

# 验证结果一致
res_direct = direct_filter_2d(test_img, k_2d)
res_sep = separable_filter_2d(test_img, k_1d)
print(f'结果差异: {np.abs(res_direct - res_sep).max():.6e} (应接近0)')

# 性能对比
n_runs = 50

t_direct = timeit.timeit(
    lambda: direct_filter_2d(test_img, k_2d), number=n_runs
)
t_sep = timeit.timeit(
    lambda: separable_filter_2d(test_img, k_1d), number=n_runs
)

print(f'\n图像大小: {test_img.shape}, 核大小: {size}x{size}')
print(f'直接 2D 卷积: {t_direct/n_runs*1000:.2f} ms/call')
print(f'可分离滤波:   {t_sep/n_runs*1000:.2f} ms/call')
print(f'加速比: {t_direct/t_sep:.2f}x')
print(f'\n理论加速比: {size**2 / (2*size):.1f}x (= K² / 2K)')

## 三、面试要点总结

| 知识点 | 要点 |
|--------|------|
| 均值 vs 高斯 | 均匀权重 vs 中心权重大 |
| 中值滤波 | 擅长去椒盐噪声, 但速度慢 |
| 双边滤波 | 保边去噪, 空间+值域双重权重, 速度最慢 |
| Sobel | 一阶导数, 可分离, 检测方向边缘 |
| Laplacian | 二阶导数, 各向同性, 对噪声敏感 |
| 可分离滤波 | 2D 拆成 2 个 1D, 复杂度 O(N²) -> O(2N) |
| filter2D | 执行互相关 (非严格卷积), 默认 BORDER_REPLICATE |
| ddepth | 梯度计算用 CV_64F 避免截断, -1 保持同类型 |